In [1]:
%pylab inline
import os
import gc
import matplotlib
import numpy as np
import astropy.io.fits as pyfits
import astropy.table as astTab
import sim_analysis_utilities as utilities

Populating the interactive namespace from numpy and matplotlib


In [2]:
cd wrkDir

/home/xiangchong/work/gw_ipmu/work/galSim-HSC


In [29]:
info=astTab.Table.read('fieldTractInfoS19.csv')
for fieldname in utilities.field_names:
    ids=astTab.Table.read('./s19/s19-1/anaCat_newS19Mask_fdeltacut/catalog_obs_reGaus_blinded_m/xiangchong-3blind-test/%s_blinded_0_m.fits' %fieldname)['object_id',]
    infoU=info[info['field']==fieldname]
    catAll=[]
    for ifo in infoU:
        sub=astTab.Table.read('./database/s19-colorterm/tracts/%d.fits' %(ifo['tract']))
        sub=utilities.delcolnull(sub)
        join=astTab.join(left=sub,right=ids,keys='object_id',join_type='inner')
        catAll.append(join)
        del sub,join
        gc.collect()
    catOut=astTab.vstack(catAll)
    catOut.sort('object_id')
    del catAll
    assert np.all(catOut['object_id']==ids['object_id']),'failed'
    catOut.write('./s19/s19-1/anaCat_newS19Mask_fdeltacut/catalog_others/%s_colorterm.fits'%fieldname,overwrite=True)
    del catOut
    gc.collect()